# Sesión 05 - IA en Python con cuaderno de Jupyer en Colaboratory

Instrucciones:

1. Para ejecutar una celda hay que utilizar ctrl+shift, lo cual generará el resultado del código más una nueva celda de programación.

2. Previamente a la ejecución del cuaderno, hay que verificar que tenemos conectadi la distribución de Python. Normalmente se aloja en la parte superior derecha de la interfaz de Google Colaboratory.

In [1]:
# Instalar paquetes necesarios

%pip install -U google groq openai ipywidgets

In [2]:
# Importaciones comunes

import os
from getpass import getpass
import ipywidgets as widgets
from IPython.display import display, clear_output

In [3]:
# Elegir proveedor y preparar la clave

proveedor = "gemini"

variables_api = {
    "gemini": "GEMINI_API_KEY",
    "groq":"GROQ_API_KEY",
    "openrouter": "OPENROUTER_API_KEY"
}

if proveedor not in variables_api:
  raise ValueError("Elige gemini, groq u openrouter.")

clave_api = os.environ.get(variables_api[proveedor], "").strip()
if not clave_api:
  clave_api = getpass(f"Clave de provedor {proveedor}: ").strip()
if not clave_api:
  raise ValueError("La clave no puede estar vacía.")

print("Proveedor seleccionado:", proveedor)

Clave de provedor gemini: ··········
Proveedor seleccionado: gemini


In [4]:
# Creación de la conexión

if proveedor == "gemini":
  from google import genai
  from google.genai import types
  cliente = genai.Client(
      api_key = clave_api,
      http_options={"timeout": 120000}
  )

elif proveedor == "groq":
    from groq import Groq
    cliente = Groq(api_key = clave_api, timeout = 120.0, max_retries=0)

else:
  from openai import OpenAI
  cliente = OpenAI(
      api_key = clave_api,
      base_url = "https://openrouter.ai/api/v1",
      timeout = 120.0,
      max_retries=0
  )

# La clave (api) sigue en la memoria interna del cliente, pero vamos a quitar esa copia
del clave_api
print("Cliente creado. El siguiente paso es consultar el catálogo")

Cliente creado. El siguiente paso es consultar el catálogo


In [5]:
# Consultar modelos disponibles

if proveedor == "gemini":
  catalogo = list(cliente.models.list())
  modelos = sorted(
      m.name.removeprefix("models/")
      for m in catalogo
      if "generateContent" in (m.supported_actions or [])
  )

else:
  catalogo = cliente.models.list()
  modelos = sorted(m.id for m in catalogo.data)

print("Modelos encontrados: ", len(modelos))
for nombre in modelos:
  print(nombre)

Modelos encontrados:  46
antigravity-preview-05-2026
antigravity-preview-09-2026
antigravity-preview-latest
deep-research-max-preview-04-2026
deep-research-preview-04-2026
deep-research-pro-preview-12-2025
gemini-2.5-computer-use-preview-10-2025
gemini-2.5-flash
gemini-2.5-flash-image
gemini-2.5-flash-lite
gemini-2.5-flash-preview-tts
gemini-2.5-pro
gemini-2.5-pro-preview-tts
gemini-3-flash-preview
gemini-3-pro-image
gemini-3-pro-image-preview
gemini-3.1-flash-image
gemini-3.1-flash-image-preview
gemini-3.1-flash-lite
gemini-3.1-flash-lite-image
gemini-3.1-flash-lite-preview
gemini-3.1-flash-tts-preview
gemini-3.1-pro-preview
gemini-3.1-pro-preview-customtools
gemini-3.5-flash
gemini-3.5-flash-lite
gemini-3.5-transcribe
gemini-3.6-flash
gemini-3.7-flash
gemini-3.8-flash
gemini-3.8-flash-lite-tts
gemini-3.8-flash-tts
gemini-4-argon
gemini-flash-latest
gemini-flash-lite-latest
gemini-nano-banana-2.1
gemini-omni-1.1-flash
gemini-omni-flash-preview
gemini-pro-latest
gemini-robotics-er-2-pr

In [6]:
# Elegir mododelo y abrir una conversación

modelo = input("ID del modelo de texto/chat: ").strip()
if not modelo:
  raise ValueError("Debes indicar un modelo.")

sistema = (
    "Eres un tutor experto en Python y análisis de datos. Responde en español, "
    "paso a paso, con código separado en celdas para Jupyter. Explica supuestos "
    "e interpretación de las salidas. No inventes datos, resultados ni ejecuciones."
)

historial = [{"role": "system", "content": sistema}]

if proveedor == "gemini":
  chat = cliente.chats.create(
      model = modelo,
      config = types.GenerateContentConfig(system_instruction = sistema),
  )
print("Conversación preparada con:", proveedor, "/", modelo)

ID del modelo de texto/chat: gemini-3.6-flash
Conversación preparada con: gemini / gemini-3.6-flash


In [7]:
# Primera consulta: Sin definir funciones

prompt = "Explica qué es un DataFrame de pandas y muestra un ejemplo con tres filas."

if proveedor == "gemini":
  respuesta = chat.send_message(prompt)
  texto = respuesta.text
else:
  mensajes = historial + [{"role": "user", "content":prompt}]
  respuesta = cliente.chat.completions.create(
      model = modelo,
      messages = mensajes
  )
  texto = respuesta.choices[0].message.content

if not texto or not texto.strip():
  raise ValueError("No se recibió texto. Revisa el modelo o el motivo de bloqueo")

if proveedor != "gemini":
  historial.extend([
      {"role": "user", "content": prompt},
      {"role": "assistant", "content":texto}
  ])

print(texto)

¡Hola! Como tu tutor de Python y análisis de datos, con gusto te explico qué es un **DataFrame** de pandas y cómo construir uno paso a paso.

---

### ¿Qué es un DataFrame de pandas?

Un **DataFrame** es la estructura de datos principal de la librería `pandas`. Puedes imaginarlo simplemente como una **tabla de datos bidimensional**, muy similar a una hoja de cálculo de Excel o a una tabla en una base de datos SQL.

Sus características principales son:
1. **Bidimensional:** Tiene filas (eje 0) y columnas (eje 1).
2. **Etiquetado:** Las filas tienen un índice (identificador) y las columnas tienen nombres.
3. **Heterogéneo:** Cada columna puede contener un tipo de dato diferente (números enteros, flotantes, texto, fechas, booleanos, etc.), pero dentro de una misma columna todos los datos suelen ser del mismo tipo.

---

### Supuestos previos
* Asumimos que estás trabajando en un entorno de **Jupyter Notebook** o **JupyterLab**.
* Asumimos que la librería `pandas` ya está instalada en tu e

Como vimos anteriormente, un **DataFrame** de Pandas es una estructura de datos bidimensional en forma de tabla (con filas y columnas), similar a una hoja de Excel o una tabla SQL.

Para ofrecerte un segundo ejemplo práctico y diferente, crearemos un DataFrame relativo a un **inventario de productos** con exactamente **tres filas**.

---

### Ejemplo práctico paso a paso (Jupyter Notebook)

#### **Celda 1: Importar la librería Pandas**
```python
import pandas as pd
```

---

#### **Celda 2: Crear el DataFrame de productos (3 filas)**
En este caso, usaremos un diccionario de Python donde cada clave es una columna y cada lista contiene los valores para 3 productos diferentes.

```python
# Definimos los datos con 3 filas por columna
datos_productos = {
    "Producto": ["Laptop", "Teclado", "Monitor"],
    "Categoría": ["Electrónica", "Accesorios", "Electrónica"],
    "Precio": [899.99, 25.50, 199.00],
    "Stock": [15, 50, 8],
}

# Creamos el DataFrame
df_inventario = pd.DataFrame(datos_productos)
```

---

#### **Celda 3: Visualizar el DataFrame**
```python
df_inventario
```

**Representación gráfica de la salida en Jupyter:**

| | Producto | Categoría | Precio | Stock |
| :--- | :--- | :--- | :--- | :--- |
| **0** | Laptop | Electrónica | 899.99 | 15 |
| **1** | Teclado | Accesorios | 25.50 | 50 |
| **2** | Monitor | Electrónica | 199.00 | 8 |

---

### Supuestos del modelo

1. **Entorno de ejecución:** Se asume que la biblioteca `pandas` está instalada en el entorno donde ejecutas el Jupyter Notebook.
2. **Homogeneidad en la longitud:** Todas las listas asociadas a las claves del diccionario deben tener la misma cantidad de elementos (en este caso, 3), de lo contrario Pandas generará un error de tipo `ValueError`.
3. **Indexación predeterminada:** Al no asignar un índice explícito (como códigos de producto), Pandas genera automáticamente una secuencia numérica entera iniciando en `0`.

---

### Interpretación de la salida

1. **Estructura de Filas e Índice (`Index`):**
   * Las filas corresponden a cada registro (Laptop, Teclado, Monitor).
   * Los números `0`, `1` y `2` forman el **índice numérico predeterminado**, permitiendo identificar de forma única la posición de cada fila.

2. **Estructura de Columnas y Tipos de Datos Inferidos:**
   * **`Producto` y `Categoría`:** Cadenas de texto. Pandas les asigna el tipo de dato `object` (o `string` en versiones recientes).
   * **`Precio`:** Números con punto decimal. Pandas asigna automáticamente el tipo `float64`.
   * **`Stock`:** Números enteros. Pandas asigna automáticamente el tipo `int64`.

3. **Dimensiones:** El DataFrame posee una forma (*shape*) de `(3, 4)`, representando 3 filas y 4 columnas en total.

In [8]:
import pandas as pd
# Definimos los datos con 3 filas por columna
datos_productos = {
    "Producto": ["Laptop", "Teclado", "Monitor"],
    "Categoría": ["Electrónica", "Accesorios", "Electrónica"],
    "Precio": [899.99, 25.50, 199.00],
    "Stock": [15, 50, 8],
}

# Creamos el DataFrame
df_inventario = pd.DataFrame(datos_productos)
df_inventario

,Producto,Categoría,Precio,Stock
0,Laptop,Electrónica,899.99,15
1,Teclado,Accesorios,25.50,50
2,Monitor,Electrónica,199.00,8


In [19]:
from google.colab import output

output.enable_custom_widget_manager()

In [13]:
# Crear el cuadro de texto

if "panel_ia" in globals():
    panel_ia.close()

entrada = widgets.Textarea(
    value="",
    placeholder="Escribe una pregunta o pega código y explica qué necesitas…",
    description="Prompt:",
    layout=widgets.Layout(width="100%", height="180px"),
)
boton = widgets.Button(
    description="Consultar IA",
    icon="paper-plane",
    button_style="warning",
    tooltip="Enviar el prompt al proveedor seleccionado",
)
salida = widgets.Output()

print("Controles preparados. Ejecuta la siguiente celda para activar el botón.")



Controles preparados. Ejecuta la siguiente celda para activar el botón.


In [14]:
# Activar el botón: la única función

def al_consultar(b):
    pregunta = entrada.value.strip()
    with salida:
        clear_output(wait=True)
        if not pregunta:
            print("Primero escribe una pregunta.")
            return

        b.disabled = True
        b.description = "Consultando…"
        try:
            if proveedor == "gemini":
                resultado = chat.send_message(pregunta)
                texto_respuesta = resultado.text
            else:
                mensajes = historial + [{"role": "user", "content": pregunta}]
                resultado = cliente.chat.completions.create(
                    model=modelo,
                    messages=mensajes,
                )
                texto_respuesta = resultado.choices[0].message.content

            if not texto_respuesta or not texto_respuesta.strip():
                raise ValueError("Respuesta sin texto.")

            if proveedor != "gemini":
                historial.extend([
                    {"role": "user", "content": pregunta},
                    {"role": "assistant", "content": texto_respuesta},
                ])
            print(texto_respuesta)

        except Exception as error:
            print("No se completó la consulta:", type(error).__name__)
            print("Revisa clave, modelo, permisos, cuota y conexión.")
        finally:
            b.disabled = False
            b.description = "Consultar IA"

In [18]:
# Evita registrar el mismo evento varias veces al repetir esta celda.
if "evento_ia" in globals():
    boton.on_click(evento_ia, remove=True)
evento_ia = al_consultar
boton.on_click(evento_ia)

if "panel_ia" in globals():
    panel_ia.close()
panel_ia = widgets.VBox([entrada, boton, salida])
display(panel_ia)